# Predictive Anayltics: Support Vector Machines with Regression

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [98]:
from run_config import PATHS

In [99]:
baseline_path = PATHS.train_test_dir / "baseline_predictions"

In [100]:
import pandas as pd
import numpy as np
import matplotlib as plt
import datetime
from joblib import load, dump
import h3
from tabulate import tabulate

import pandas as pd
import numpy as np
#import matplotlib as plt
import datetime

from sklearn.kernel_approximation import Nystroem
from joblib import load, dump
import h3
from joblib import Memory
import jinja2

In [101]:
INPUT =  f"../models/svm/"

In [102]:
def met(own_results, baseline):
    score = 1 - (own_results/baseline)
    return score
    

### load y_pred and models

In [103]:
SPATIAL_UNITS = ["HEXAGON_7", "HEXAGON_8", "CENSUS_TRACTS", "COMMUNITY_AREAS"]
TIME_UNITS = ["1H", "4H", "24H"]

In [104]:
path_csv = {
    (spatial, time): f"{INPUT}result_{spatial}_{time}.csv"
    for spatial in SPATIAL_UNITS
    for time in TIME_UNITS
}

In [105]:
results = {
    (spatial, time): pd.read_csv(path)
    for (spatial, time), path in path_csv.items()
}

In [106]:
combined = pd.concat(
    [df.assign(spatial=spatial, time=time) for (spatial, time), df in results.items()],
    ignore_index=True
)
combined.to_csv("../results/results_svr.csv", index=False)

In [ ]:
# table 
results_df = combined[["spatial", "time", "MAE", "MSE", "RMSE", "R2 Score"]]
results_df = results_df.sort_values(["spatial", "time"]).reset_index(drop=True)

styled = results_df.style.format({
    "MAE": "{:.2f}",-
    "MSE": "{:.2f}",
    "RMSE": "{:.2f}",
    "R2 Score": "{:.3f}",
}).background_gradient(subset=["R2 Score"], cmap="RdYlGn", vmin=0, vmax=1) \
  .background_gradient(subset=["MAE", "RMSE", "MSE"], cmap="RdYlGn_r") \
  .set_caption("SVM Regression Results by Spatial Unit and Time Granularity")

styled

,spatial,time,MAE,MSE,RMSE,R2 Score
0,CENSUS_TRACTS,1H,0.36,9.57,3.09,0.548
1,CENSUS_TRACTS,24H,10.60,3097.68,55.66,0.541
2,CENSUS_TRACTS,4H,1.35,127.07,11.27,0.585
3,COMMUNITY_AREAS,1H,2.38,120.04,10.96,0.785
4,COMMUNITY_AREAS,24H,44.69,26132.86,161.66,0.855
5,COMMUNITY_AREAS,4H,8.46,1140.26,33.77,0.861
6,HEXAGON_7,1H,1.63,61.14,7.82,0.793
7,HEXAGON_7,24H,26.10,6547.67,80.92,0.934
8,HEXAGON_7,4H,5.53,567.02,23.81,0.871
9,HEXAGON_8,1H,0.38,10.85,3.29,0.522


In [108]:
path_y_pred = {
    (spatial, time): f"{INPUT}model_{spatial}_{time}.csv"
    for spatial in SPATIAL_UNITS
    for time in TIME_UNITS
}

In [109]:
models = {
    (spatial, time): pd.read_csv(path)
    for (spatial, time), path in path_y_pred.items()
}

In [110]:
models

{('HEXAGON_7',
  '1H'):           y_pred  y_test          h3_cell                 date
 0       1.947378       0  872664c8effffff  2026-01-13 01:00:00
 1       0.995485       0  872664c8effffff  2025-11-30 01:00:00
 2       0.000000       0  872664c13ffffff  2025-07-31 14:00:00
 3       0.000000       0  872664c8effffff  2025-07-31 14:00:00
 4       0.000000       0  872664c8effffff  2025-06-29 06:00:00
 ...          ...     ...              ...                  ...
 285571  0.565287       0  872664d9effffff  2025-06-01 23:00:00
 285572  0.000000       0  872664d9effffff  2025-08-13 22:00:00
 285573  1.055305       0  872664d9effffff  2025-07-14 06:00:00
 285574  2.518632       0  872664d9effffff  2025-01-11 06:00:00
 285575  0.000000       0  872664d9effffff  2025-01-11 15:00:00
 
 [285576 rows x 4 columns],
 ('HEXAGON_7',
  '4H'):            y_pred  y_test          h3_cell                 date
 0        0.000000       0  872664c8cffffff  2025-08-12 20:00:00
 1        3.821714       0

In [111]:
baseline = pd.read_parquet(f"{baseline_path}/baseline_test_metrics.parquet")

In [112]:
baseline

,dataset,baseline,metric,value
0,hexagon_h3r7_1h,zero,n_samples,285576.000000
1,hexagon_h3r7_1h,zero,mae,1.936539
2,hexagon_h3r7_1h,zero,mse,299.480387
3,hexagon_h3r7_1h,zero,rmse,17.305502
4,hexagon_h3r7_1h,zero,r2,-0.012681
...,...,...,...,...
571,community_areas_unfiltered_4h,spatial_time_weekday_mean,nonzero_rmse,38.622753
572,community_areas_unfiltered_4h,spatial_time_weekday_mean,peak_threshold,63.000000
573,community_areas_unfiltered_4h,spatial_time_weekday_mean,peak_share,0.081065
574,community_areas_unfiltered_4h,spatial_time_weekday_mean,peak_mae,66.392831


In [135]:
results_df["spatial_clean"] = (
    results_df["spatial"].str.lower()
    .str.replace("hexagon_", "hexagon_h3r", regex=False)  # HEXAGON_7 -> hexagon_h3r7
)
results_df["time_clean"] = results_df["time"].str.lower()  # 1H -> 1h
results_df["run"] = results_df["spatial_clean"] + "_" + results_df["time_clean"]

own_results = results_df.melt(
    id_vars=["dataset"],
    value_vars=["MAE", "MSE", "RMSE", "R2 Score"],
    var_name="metric",
    value_name="value",
)

own_results["metric"] = own_results["metric"].str.lower().str.replace(" score", "", regex=False)

In [138]:
baseline_mae = baseline[baseline["metric"] == "mae"][["dataset", "metric", "value"]]
own_mae = own_results[own_results["metric"] == "mae"][["dataset", "metric", "value"]]


In [139]:

merged = own_mae.merge(
    baseline_mae,
    on="dataset",
    suffixes=("_own", "_baseline")
)
merged


,dataset,metric_own,value_own,metric_baseline,value_baseline
0,census_tracts_1h,mae,0.359323,mae,0.359517
1,census_tracts_1h,mae,0.359323,mae,0.359517
2,census_tracts_1h,mae,0.359323,mae,0.135651
3,census_tracts_24h,mae,10.595766,mae,8.628405
4,census_tracts_24h,mae,10.595766,mae,8.628405
5,census_tracts_24h,mae,10.595766,mae,2.175618
6,census_tracts_4h,mae,1.349104,mae,1.438068
7,census_tracts_4h,mae,1.349104,mae,1.438068
8,census_tracts_4h,mae,1.349104,mae,0.439747
9,community_areas_1h,mae,2.378622,mae,4.099426


In [142]:
merged["score"] = met(merged["metric_own"], merged["metric_baseline"])

TypeError: unsupported operand type(s) for /: 'str' and 'str'